# 02 · Build a cube from a tidy table

Field observations often arrive as one row per time and location. A complete
time × y × x table can become a cube by indexing its coordinate columns and
using xarray. We then call a verb directly and plot what standardization did.

**Pattern:** tidy `DataFrame` → indexed xarray cube → direct verb call.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cubedynamics import verbs as v

# Build every time/location combination. Real field data would usually be read
# from CSV or Parquet, but the following table has the same tidy structure.
time = pd.date_range("2025-06-01", periods=14, freq="D")
y = [40.2, 40.0, 39.8]
x = [-105.2, -105.0, -104.8, -104.6]
index = pd.MultiIndex.from_product([time, y, x], names=["time", "y", "x"])
table = index.to_frame(index=False)

# Create a deterministic weekly cycle plus small latitude/longitude effects.
# Vectorized columns keep the recipe close to ordinary pandas workflows.
day = (table["time"] - time[0]).dt.days.to_numpy()
table["temperature"] = (
    27
    + 5 * np.sin(2 * np.pi * day / 7)
    + 1.5 * (table["y"].to_numpy() - 40)
    - 0.8 * (table["x"].to_numpy() + 105)
)

# Indexing by time, y, and x tells xarray which columns define cube axes.
# transpose makes the conventional (time, y, x) order explicit.
cube = (
    table.set_index(["time", "y", "x"])
    .to_xarray()["temperature"]
    .transpose("time", "y", "x")
)
cube.name = "temperature"
cube.attrs.update(units="degC", source="synthetic tidy table")

# Direct verb style: configure zscore first, then pass the cube to the returned
# callable. Each pixel is standardized independently along time.
standardized = v.zscore(dim="time")(cube)

# A temporal z-score should have mean zero at every location (up to rounding).
assert cube.shape == (14, 3, 4)
assert abs(float(standardized.mean("time").max())) < 1e-12

# Compare the same coordinate before and after the verb so its effect is clear.
site = {"y": 40.0, "x": -105.0}
fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True, constrained_layout=True)
cube.sel(**site).plot(ax=axes[0], marker="o", color="#8b543c")
axes[0].set_title("Table values after reshaping to a cube")
axes[0].set_ylabel("Temperature (°C)")
standardized.sel(**site).plot(ax=axes[1], marker="o", color="#3f6f72")
axes[1].axhline(0, color="0.35", linewidth=0.8)
axes[1].set_title("The same pixel after v.zscore(dim='time')")
axes[1].set_ylabel("Standard deviations")
plt.show()